In [ ]:
import os
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp as sivp
from numpy import sin, cos, exp, sqrt, mod, heaviside
import pandas as pd
import numpy as np

In [ ]:
# See if csv is there
print(os.getcwd())
print(os.listdir())

# FOR TERMINAL:

# dx download "/T2D/t2d_pilot_100.csv"

In [ ]:
df_raw = pd.read_csv("t2d_diabetics_full.csv")
df_h_raw = pd.read_csv("t2d_pilot_healthy_full.csv")

In [ ]:
# Strip prefix
df_raw.columns = [c.replace('participant.', '') for c in df_raw.columns]

# Mandatory columns for the Aggressive Drop (Biomarkers + 16 Mental Health questions)
stress_cols = ['p20510', 'p20507', 'p20519', 'p20514', 'p20511', 'p20513', 'p20508', 'p20518', 
               'p20505', 'p20512', 'p20506', 'p20509', 'p20516', 'p20515', 'p20520', 'p20517']
biomarker_cols = ['p30740_i0', 'p21001_i0', 'p31', 'p21022']
med_cols = [f'p20003_i0_a{i}' for i in range(8)]

# Drop only if required biomarker or stress columns are missing
# med_cols are allowed to be NaN (empty slots = no medication, not missing data)
required_cols = biomarker_cols + stress_cols
df_raw[med_cols] = df_raw[med_cols].fillna(0)
df = df_raw.dropna(subset=required_cols)

# Filter for valid UKB stress scores (1-4)
for col in stress_cols:
    df = df[df[col] >= 1]

# Transform 1-4 scale to 0-3 and normalize to 0-1 (16 questions * max score of 3 = 48)
df['S_input'] = (df[stress_cols] - 1).sum(axis=1) / 48.0

# Medication Wall (Exclude Metformin and Insulin)
treated_codes = [1140884600, 1140871310, 1140874686, 1140883066]
is_treated = df[med_cols].isin(treated_codes).any(axis=1)
df_untreated = df[~is_treated].copy()

# Convert Glucose to mg/dL
df_untreated['G_mgdl'] = df_untreated['p30740_i0'] * 18.0182

# Final filter for the first 50 strictly untreated diabetics (>126)
df_diabetic_final = df_untreated[df_untreated['G_mgdl'] >= 126].head(50)

print(f"Aggressive Drop complete. Final Diabetic N: {len(df_diabetic_final)}")

In [ ]:
# Save the pilot 50:
df_diabetic_final.to_csv("t2d_pilot_50_diabetics.csv", index=False)

In [ ]:
# Strip prefix
df_h_raw.columns = [c.replace('participant.', '') for c in df_h_raw.columns]

# Drop only if required columns are missing
# med_cols NaN = empty medication slot, fill with 0
df_h_raw[med_cols] = df_h_raw[med_cols].fillna(0)
df_h = df_h_raw.dropna(subset=biomarker_cols + stress_cols)

# Filter for valid 1-4 range
for col in stress_cols:
    df_h = df_h[df_h[col] >= 1]

# Same 0-3 scale math and normalization
df_h['S_input'] = (df_h[stress_cols] - 1).sum(axis=1) / 48.0

# Medication Wall
is_treated_h = df_h[med_cols].isin(treated_codes).any(axis=1)
df_h_untreated = df_h[~is_treated_h].copy()

# Glucose to mg/dL
df_h_untreated['G_mgdl'] = df_h_untreated['p30740_i0'] * 18.0182

# Final filter for first 50 strictly untreated healthy controls (70-100)
df_healthy_final = df_h_untreated[(df_h_untreated['G_mgdl'] >= 70) & (df_h_untreated['G_mgdl'] < 100)].head(50)

print(f"Aggressive Drop complete. Final Healthy N: {len(df_healthy_final)}")

In [ ]:
# Save the pilot 50:
df_healthy_final.to_csv("t2d_pilot_50_healthy.csv", index=False)